# TP3 : Traitement d'images mammographiques

Notebook converti depuis `tp_mammographie.py`. Exécuter les cellules dans l'ordre. Placez `mammogram.png` dans le même dossier que ce notebook.

In [ ]:
import os
import heapq
from collections import Counter

import numpy as np
import cv2
import matplotlib.pyplot as plt

OUTPUT_DIR = "image_resultat"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# Partie 1 : chargement et analyse de l'image
def load_image(filename):
    img = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Image not found: {filename}")
    return img


def show_image(img, title="Image"):
    plt.figure(figsize=(6, 6))
    plt.imshow(img, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()


def analyze_histogram(img, title="Analyse de l'image"):
    """Affiche les statistiques basiques et l'histogramme de l'image."""
    print(f"--- Analyse Statistiques de {title} ---")
    print(f"Min: {img.min()}, Max: {img.max()}, Moyenne: {img.mean():.2f}")

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(img, cmap="gray")
    plt.title(title)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.hist(img.ravel(), bins=256, range=[0, 256], color="black")
    plt.title("Histogramme")
    plt.xlabel(f"Niveaux de gris \nMin: {img.min()} | Max: {img.max()} | Moy: {img.mean():.1f}")
    plt.ylabel("Fréquence")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "partie1_histogramme.png"))
    plt.show()


# Partie 2 : segmentation et transformations d'histogramme
def dynamic_expansion(img, new_min=0, new_max=255):
    """Expansion de dynamique (étirement linéaire de l'histogramme)."""
    i_min, i_max = float(img.min()), float(img.max())
    if i_max == i_min:
        return img.copy()
    expanded = (img - i_min) * ((new_max - new_min) / (i_max - i_min)) + new_min
    return np.clip(expanded, new_min, new_max).astype(np.uint8)


def histogram_equalization(img):
    """Égalisation globale basée sur la fonction de répartition cumulative (CDF)."""
    hist, bins = np.histogram(img.flatten(), 256, [0, 256])
    cdf = hist.cumsum()
    cdf_m = np.ma.masked_equal(cdf, 0)
    cdf_m = (cdf_m - cdf_m.min()) * 255 / (cdf_m.max() - cdf_m.min())
    cdf = np.ma.filled(cdf_m, 0).astype("uint8")
    return cdf[img]


def histogram_specification(img, reference):
    """Mise en correspondance d'histogramme (spécification)."""
    hist_src, _ = np.histogram(img.flatten(), 256, [0, 256])
    hist_ref, _ = np.histogram(reference.flatten(), 256, [0, 256])

    cdf_src = hist_src.cumsum() / hist_src.sum()
    cdf_ref = hist_ref.cumsum() / hist_ref.sum()

    lut = np.zeros(256, dtype=np.uint8)
    for gray_level in range(256):
        diff = np.abs(cdf_src[gray_level] - cdf_ref)
        lut[gray_level] = np.argmin(diff)

    return lut[img]


# Partie 3 : réduction du bruit
def add_gaussian_noise(img, mean=0, sigma=25, seed=42):
    """Ajoute un bruit gaussien à l'image."""
    np.random.seed(seed)
    gauss = np.random.normal(mean, sigma, img.shape)
    noisy = img.astype(np.float32) + gauss
    return np.clip(noisy, 0, 255).astype(np.uint8)


def add_salt_and_pepper_noise(img, amount=0.05, seed=42):
    """Ajoute un bruit impulsionnel (sel et poivre)."""
    np.random.seed(seed)
    noisy = img.copy()
    num_salt = np.ceil(amount * img.size * 0.5)
    num_pepper = np.ceil(amount * img.size * 0.5)

    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in img.shape]
    noisy[tuple(coords)] = 255
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in img.shape]
    noisy[tuple(coords)] = 0
    return noisy


def mean_filter(img, ksize=3):
    """Filtre moyenneur fait maison."""
    pad = ksize // 2
    padded = np.pad(img, pad, mode="reflect")
    output = np.zeros_like(img, dtype=np.float32)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i : i + ksize, j : j + ksize]
            output[i, j] = np.mean(region)
    return output.astype(np.uint8)


def gaussian_filter(img, ksize=3, sigma=1.0):
    """Filtre gaussien fait maison."""
    pad = ksize // 2
    ax = np.linspace(-(ksize // 2), ksize // 2, ksize)
    gauss = np.exp(-0.5 * np.square(ax) / np.square(sigma))
    kernel = np.outer(gauss, gauss)
    kernel /= kernel.sum()

    padded = np.pad(img, pad, mode="reflect")
    output = np.zeros_like(img, dtype=np.float32)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i : i + ksize, j : j + ksize]
            output[i, j] = np.sum(region * kernel)
    return output.astype(np.uint8)


def median_filter(img, ksize=3):
    """Filtre médian fait maison."""
    pad = ksize // 2
    padded = np.pad(img, pad, mode="reflect")
    output = np.zeros_like(img, dtype=np.uint8)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i : i + ksize, j : j + ksize]
            output[i, j] = np.median(region)
    return output


# Partie 4 : binarisation et morphologie
def binarize_image(img, threshold):
    """Binarisation de l'image selon un seuil donné."""
    return np.where(img >= threshold, 255, 0).astype(np.uint8)


def erosion(img, kernel):
    """Érosion morphologique."""
    k_h, k_w = kernel.shape
    pad_h, pad_w = k_h // 2, k_w // 2
    padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode="constant", constant_values=0)
    output = np.zeros_like(img)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i : i + k_h, j : j + k_w]
            if np.all(region[kernel == 1] == 255):
                output[i, j] = 255
    return output


def dilation(img, kernel):
    """Dilatation morphologique."""
    k_h, k_w = kernel.shape
    pad_h, pad_w = k_h // 2, k_w // 2
    padded = np.pad(img, ((pad_h, pad_h), (pad_w, pad_w)), mode="constant", constant_values=0)
    output = np.zeros_like(img)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            region = padded[i : i + k_h, j : j + k_w]
            if np.any(region[kernel == 1] == 255):
                output[i, j] = 255
    return output


def opening(img, kernel):
    """Ouverture morphologique (érosion puis dilatation)."""
    return dilation(erosion(img, kernel), kernel)


def closing(img, kernel):
    """Fermeture morphologique (dilatation puis érosion)."""
    return erosion(dilation(img, kernel), kernel)


# Partie 5 : détection de bords et rehaussement
def sobel_edges(img):
    """Détection de contours par filtre de Sobel (magnitude du gradient)."""
    sobel_x = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
    magnitude = np.sqrt(sobel_x**2 + sobel_y**2)
    return np.clip(magnitude, 0, 255).astype(np.uint8)


def laplacian_filter(img):
    """Filtrage laplacien pour accentuer les variations locales de contraste."""
    lap = cv2.Laplacian(img, cv2.CV_64F)
    return np.clip(np.abs(lap), 0, 255).astype(np.uint8)


# Partie 6 : évaluation quantitative
def mse(img1, img2):
    """Calcule l'Erreur Quadratique Moyenne (Mean Squared Error)."""
    return np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)


def psnr(img1, img2):
    """Calcule le Rapport Signal sur Bruit de Crête."""
    err = mse(img1, img2)
    if err == 0:
        return float("inf")
    return 10.0 * np.log10((255.0**2) / err)


# Partie 7 : égalisation adaptative locale
def apply_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
    """Applique l'égalisation d'histogramme adaptative limitée en contraste."""
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(img)


# Partie 8 : compression sans perte
def huffman_encoding(img):
    """Effectue l'encodage de Huffman et calcule la longueur moyenne des codes."""
    flat_img = img.flatten()
    total_pixels = len(flat_img)
    if total_pixels == 0:
        raise ValueError("Cannot encode an empty image.")
    freqs = Counter(flat_img)
    root = build_huffman_tree(freqs)
    codes = generate_huffman_codes(root)
    encoded_bits = "".join(codes[pixel] for pixel in flat_img)
    avg_len = sum((freq / total_pixels) * len(codes[symbol]) for symbol, freq in freqs.items())
    return encoded_bits, codes, avg_len


class Node:
    def __init__(self, symbol=None, freq=0, left=None, right=None):
        self.symbol = symbol
        self.freq = freq
        self.left = left
        self.right = right

    def __lt__(self, other):
        return self.freq < other.freq


def build_huffman_tree(frequencies):
    """Construit l'arbre de Huffman à partir des fréquences des symboles."""
    heap = [Node(symbol, freq) for symbol, freq in frequencies.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        node1 = heapq.heappop(heap)
        node2 = heapq.heappop(heap)
        merged = Node(freq=node1.freq + node2.freq, left=node1, right=node2)
        heapq.heappush(heap, merged)
    return heap[0] if heap else None


def generate_huffman_codes(root, current_code="", codes=None):
    """Génère les codes binaires pour chaque niveau de gris."""
    if codes is None:
        codes = {}
    if root is not None:
        if root.symbol is not None:
            codes[root.symbol] = current_code or "0"
        generate_huffman_codes(root.left, current_code + "0", codes)
        generate_huffman_codes(root.right, current_code + "1", codes)
    return codes

## Exécution des traitements

Les cellules suivantes reprennent les exemples du script dans l'ordre.

### Partie 1 : chargement et analyse

In [ ]:
img = load_image("mammogram.png")
analyze_histogram(img)

### Partie 2 : transformations d'histogramme

In [ ]:
dynamic_img = dynamic_expansion(img)
analyze_histogram(dynamic_img, title="Image après Expansion de Dynamique")

equalized_img = histogram_equalization(img)
analyze_histogram(equalized_img, title="Image après Égalisation d'Histograme")

specified_img = histogram_specification(img, dynamic_img)
analyze_histogram(specified_img, title="Image après Spécification d'Histograme")

### Partie 3 : réduction du bruit

In [ ]:
gaussian_noisy_img = add_gaussian_noise(img)
analyze_histogram(gaussian_noisy_img, title="Image avec Bruit Gaussien")

salt_pepper_noisy_img = add_salt_and_pepper_noise(img)
analyze_histogram(salt_pepper_noisy_img, title="Image avec Bruit Sel et Poivre")

mean_filtered_img = mean_filter(gaussian_noisy_img)
analyze_histogram(mean_filtered_img, title="Image après Filtre Moyenneur")

gaussian_filtered_img = gaussian_filter(gaussian_noisy_img)
analyze_histogram(gaussian_filtered_img, title="Image après Filtre Gaussien")

median_filtered_img = median_filter(salt_pepper_noisy_img)
analyze_histogram(median_filtered_img, title="Image après Filtre Médian")

### Partie 4 : binarisation et morphologie

In [ ]:
binarized_img = binarize_image(equalized_img, threshold=128)
analyze_histogram(binarized_img, title="Image après Binarisation")

erosion_kernel = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]], dtype=np.uint8)
eroded_img = erosion(binarized_img, erosion_kernel)
analyze_histogram(eroded_img, title="Image après Érosion")

dilated_img = dilation(binarized_img, erosion_kernel)
analyze_histogram(dilated_img, title="Image après Dilatation")

opened_img = opening(binarized_img, erosion_kernel)
analyze_histogram(opened_img, title="Image après Ouverture")

closed_img = closing(binarized_img, erosion_kernel)
analyze_histogram(closed_img, title="Image après Fermeture")

### Partie 5 : détection de bords et rehaussement

In [ ]:
sobel_img = sobel_edges(equalized_img)
analyze_histogram(sobel_img, title="Image après Détection de Contours (Sobel)")

laplacian_img = laplacian_filter(equalized_img)
analyze_histogram(laplacian_img, title="Image après Filtrage Laplacien")

### Partie 6 : évaluation quantitative

In [ ]:
mse_value = mse(img, equalized_img)
psnr_value = psnr(img, equalized_img)
print(f"MSE : {mse_value:.2f}")
print(f"PSNR : {psnr_value:.2f} dB")

### Partie 7 : égalisation adaptative locale (CLAHE)

In [ ]:
cached_image = apply_clahe(img)
analyze_histogram(cached_image, title="Image après CLAHE")

### Partie 8 : compression sans perte (Huffman)

In [ ]:
encoded_bits, huffman_codes, average_code_length = huffman_encoding(img)
print("--- Encodage de Huffman ---")
print(f"Nombre de pixels : {img.size}")
print(f"Nombre de niveaux de gris : {len(huffman_codes)}")
print(f"Longueur moyenne : {average_code_length:.2f} bits/pixel")
print(f"Taille encodée : {len(encoded_bits)} bits")
print(f"Taille initiale : {img.size * 8} bits")

# Partie 9 : discussion et conclusion

## Réponses synthétiques aux questions de discussion

- **Traitement qui améliore le plus la lisibilité :** CLAHE (égalisation adaptative locale) est généralement efficace car elle rehausse le contraste localement, tout en limitant la surexposition.
- **Filtre qui préserve le mieux les structures fines :** le filtre médian réduit le bruit impulsionnel sans trop flouter les contours nets des microcalcifications. Le filtre gaussien peut aussi préserver davantage de détails que le filtre moyenneur.
- **Limites sur les images très bruitées :** appliquer directement un rehaussement de contraste à une image non filtrée peut amplifier le bruit et créer de fausses détections.

## Pipeline recommandé pour la détection d'anomalies

1. **Filtrage médian (3×3)** pour supprimer le bruit impulsionnel en préservant les détails.
2. **CLAHE** (`clipLimit=2.0`, `tileGridSize=(8, 8)`) pour accentuer le contraste local.
3. **Détection de contours (Sobel)** pour extraire la magnitude du gradient.
4. **Binarisation et ouverture morphologique** pour isoler les régions d'intérêt et réduire les artefacts.
5. **Évaluation (MSE / PSNR)** pour quantifier la qualité du prétraitement avant classification.